# Jour 2 — Spark & Delta Lake : couche Silver Olist (corrigé)

Notebook prêt pour le cluster Docker HDFS (`namenode:9000`). Exécuter les cellules dans l’ordre. **Attention :** les valeurs `review_score` invalides sont diagnostiquées et exclues, mais une ingestion Bronze incorrecte doit être réparée dans J1 pour restaurer les vraies notes.

In [9]:
from pyspark.sql import SparkSession, functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("J2_Spark_Delta_Silver_CORRIGE").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")
HDFS = "hdfs://namenode:9000"
BRONZE_ROOT = f"{HDFS}/lakehouse/bronze"
SILVER_MAIN_PATH = f"{HDFS}/lakehouse/silver/orders_enriched"
print("Spark prêt — UI : http://localhost:4040 (ou 4041 si 4040 occupé)")

Spark prêt — UI : http://localhost:4040 (ou 4041 si 4040 occupé)


26/11/21 14:30:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Chargement Bronze et profilage

In [10]:
BRONZE_TABLES = {"main":"orders", "customers":"customers", "items":"order_items", "products":"products", "payments":"payments", "reviews":"reviews"}
bronze_dfs = {}
for alias, name in BRONZE_TABLES.items():
    bronze_dfs[alias] = spark.read.format("delta").load(f"{BRONZE_ROOT}/{name}")
    print(f"[BRONZE OK] {name}: {bronze_dfs[alias].count():,} lignes")
orders, customers = bronze_dfs["main"], bronze_dfs["customers"]
order_items, products = bronze_dfs["items"], bronze_dfs["products"]

[BRONZE OK] orders: 99,441 lignes


[BRONZE OK] customers: 99,441 lignes


[BRONZE OK] order_items: 112,650 lignes
[BRONZE OK] products: 32,951 lignes


[BRONZE OK] payments: 103,886 lignes
[BRONZE OK] reviews: 104,162 lignes


In [11]:
print("=== SCHÉMA ORDERS ===")
orders.printSchema()
orders.select([F.sum(F.col(c).isNull().cast("long")).alias(c) for c in orders.columns]).show(truncate=False)
print("Doublons order_id :", orders.count() - orders.dropDuplicates(["order_id"]).count())

=== SCHÉMA ORDERS ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_file: string (nullable = true)



+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+---------------+------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_ingested_at|_source_dataset|_source_file|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+------------+---------------+------------+
|0       |0          |0           |0                       |160              |1783                        |2965                         |0                            |0           |0              |0           |
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------

[Stage 339:==========================================>              (3 + 1) / 4]

Doublons order_id : 0


## 2. Diagnostic des avis Bronze

Si beaucoup de notes sont invalides, cela signale un probable décalage de colonnes dans le CSV Bronze. Le filtrage ci-dessous évite le crash, mais ne répare pas la source.

In [12]:
reviews_raw = bronze_dfs["reviews"]
reviews_raw.select("review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp").show(10, truncate=False)
reviews_raw.printSchema()
reviews_checked = reviews_raw.withColumn("review_score_numeric", F.expr("try_cast(review_score AS DOUBLE)"))
valid_score = F.col("review_score_numeric").between(1, 5)
review_quality = reviews_checked.agg(
    F.count("*").alias("total_reviews"),
    F.sum(F.when(valid_score, 1).otherwise(0)).alias("valid_reviews"),
    F.sum(F.when(~F.coalesce(valid_score, F.lit(False)), 1).otherwise(0)).alias("invalid_reviews")
)
review_quality.show()
reviews_checked.filter(~F.coalesce(valid_score, F.lit(False))).select("review_id", "order_id", "review_score").show(10, truncate=False)
invalid_count = review_quality.first()["invalid_reviews"]
if invalid_count:
    print(f"ATTENTION : {invalid_count:,} notes invalides dans Bronze. Vérifier/corriger J1 avant de valider les KPI de satisfaction.")

+--------------------------------+--------------------------------+-------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score       |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+-------------------+--------------------+-----------------------+
|77e4c11fcfe64c15dba683178b461a7b|321ebafc185998f3acb39280b759b177|5                  |2017-07-07 00:00:00 |2017-07-08 11:39:35    |
|c5cef741ecac225e8ec7d915be82d5e7|c24a00c414738de9ee6121f179ebba9c|5                  |NULL                |NULL                   |
| Muito bom."                    |2017-06-02 00:00:00             |2017-06-04 00:54:19|NULL                |NULL                   |
|d75d46e5cf89d8bd89a2c6576b03f8d9|fa94f0487376b26326f5fa63d24c9a8b|5                  |2017-11-28 00:00:00 |2017-11-29 11:45:50    |
|ad866a384cdb53ae3c072116d399e97b|f9649c12307baa46d80875cd558b3e36|5 

+-------------+-------------+---------------+
|total_reviews|valid_reviews|invalid_reviews|
+-------------+-------------+---------------+
|       104162|        99224|           4938|
+-------------+-------------+---------------+



+---------------------------------------------------+-------------------+-------------------+
|review_id                                          |order_id           |review_score       |
+---------------------------------------------------+-------------------+-------------------+
|,2018-02-16 00:00:00,2018-02-20 10:52:22           |NULL               |NULL               |
|A entrega foi efetuada muito antes do prazo dado.  |NULL               |NULL               |
|O produto já começou a ser usado e até o presente  |NULL               |NULL               |
|sem problemas."                                    |2018-03-23 00:00:00|2018-04-01 00:27:51|
|Faltou Controle Remoto para Ar Condicionado Consul"|2018-01-28 00:00:00|2018-02-04 11:06:15|
| Estou satisfeita                                  |NULL               |NULL               |
|,2017-06-02 00:00:00,2017-06-03 13:12:07           |NULL               |NULL               |
|,2017-08-09 00:00:00,2017-08-09 12:03:45           |NULL   

ATTENTION : 4,938 notes invalides dans Bronze. Vérifier/corriger J1 avant de valider les KPI de satisfaction.


## 3. Nettoyage, agrégations et jointures

In [13]:
orders_clean = (orders.filter(F.col("order_id").isNotNull()).dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date")))
customers_clean = (customers.filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"))
print("Orders nettoyées :", orders_clean.count())

[Stage 368:==========================================>              (3 + 1) / 4]

Orders nettoyées : 99441


In [14]:
items = (order_items.filter(F.col("order_id").isNotNull()).groupBy("order_id")
    .agg(F.count("*").alias("item_count"),
         F.sum(F.col("price").cast("double")).alias("items_value"),
         F.sum(F.col("freight_value").cast("double")).alias("freight_value"),
         F.first("product_id", ignorenulls=True).alias("sample_product_id")))
products_clean = (products.filter(F.col("product_id").isNotNull())
    .select("product_id", "product_category_name").dropDuplicates(["product_id"]))
items_with_category = (order_items.select("order_id", "product_id")
    .join(F.broadcast(products_clean), "product_id", "left")
    .groupBy("order_id")
    .agg(F.first("product_category_name", ignorenulls=True).alias("product_category")))
payments = (bronze_dfs["payments"].filter(F.col("order_id").isNotNull())
    .groupBy("order_id")
    .agg(F.sum(F.col("payment_value").cast("double")).alias("payment_value")))
# CORRECTION : conversion tolérante + validation métier, jamais cast direct d'une date en DOUBLE
reviews = (reviews_checked.filter(F.col("order_id").isNotNull() & valid_score)
    .groupBy("order_id")
    .agg(F.avg("review_score_numeric").alias("review_score")))
print("Agrégations préparées")

Agrégations préparées


In [15]:
df_silver = (orders_clean
    .join(F.broadcast(customers_clean), "customer_id", "left")
    .join(items, "order_id", "left")
    .join(items_with_category, "order_id", "left")
    .join(payments, "order_id", "left")
    .join(reviews, "order_id", "left")
    .withColumn("delivery_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0)
    .withColumn("estimated_days", (F.unix_timestamp("order_estimated_delivery_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0)
    .withColumn("delay_days", (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_estimated_delivery_date")) / 86400.0)
    .withColumn("is_late", F.when(F.col("delay_days").isNull(), F.lit(None).cast("int"))
                .otherwise((F.col("delay_days") > 0).cast("int")))
    .withColumn("order_value", F.coalesce(F.col("payment_value"), F.col("items_value")))
    .withColumn("purchase_date", F.to_date("order_purchase_timestamp"))
    .withColumn("purchase_month", F.date_format("order_purchase_timestamp", "yyyy-MM")))
print("Silver construit")

Silver construit


## 4. Contrôles de cardinalité et match rate

In [16]:
count_before = orders_clean.count()
count_after = df_silver.count()
print("Commandes avant enrichissement :", count_before)
print("Commandes après enrichissement :", count_after)
assert count_after == count_before, "Duplication/perte de commandes après jointure"
assert df_silver.select("order_id").distinct().count() == count_after, "Doublons order_id en Silver"
match = df_silver.agg(F.count("*").alias("total"),
    F.count("customer_unique_id").alias("customers_matched"),
    F.count("item_count").alias("items_matched"),
    F.count("payment_value").alias("payments_matched"),
    F.count("review_score").alias("reviews_matched")).first()
for name in ("customers_matched", "items_matched", "payments_matched", "reviews_matched"):
    print(f"{name}: {match[name]}/{match['total']} ({100 * match[name]/match['total']:.2f}%)")
df_silver.explain("simple")

Commandes avant enrichissement : 99441
Commandes après enrichissement : 99441


customers_matched: 99441/99441 (100.00%)
items_matched: 98666/99441 (99.22%)
payments_matched: 99440/99441 (100.00%)
reviews_matched: 98673/99441 (99.23%)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#9661, customer_id#12413, order_status#12415, order_purchase_timestamp#11372, order_approved_at#11373, order_delivered_carrier_date#11374, order_delivered_customer_date#11375, order_estimated_delivery_date#11376, _ingested_at#12427, _source_dataset#12429, _source_file#12431, customer_unique_id#12433, customer_city#12437, customer_state#12439, customer_zip_code_prefix#12435, item_count#11484L, items_value#11485, freight_value#11486, sample_product_id#11487, product_category#11503, payment_value#11509, review_score#11519, delivery_days#11543, estimated_days#11544, delay_days#11545, ... 4 more fields]
   +- Project [order_id#9661, customer_id#12413, order_status#12415, order_purchase_timestamp#11372, order_approved_at#11373, order_delivered_carrier_date#11374, order_delivered_customer_date#11375, order_estimated_delivery_date#11376, _ingested_at#12427, _source_dataset#12429, _source_file#12431, customer_unique_id#

## 5. Écriture Delta Silver

In [17]:
(df_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").save(SILVER_MAIN_PATH))
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
silver_count = silver.count()
assert silver_count == count_before, "Cardinalité incorrecte après sauvegarde"
print(f"[SILVER OK] {SILVER_MAIN_PATH} — {silver_count:,} lignes")
silver.select("order_id", "customer_state", "item_count", "order_value", "review_score", "delivery_days", "delay_days", "is_late").show(10, truncate=False)

[SILVER OK] hdfs://namenode:9000/lakehouse/silver/orders_enriched — 99,441 lignes


+--------------------------------+--------------+----------+-----------------+------------+------------------+-------------------+-------+
|order_id                        |customer_state|item_count|order_value      |review_score|delivery_days     |delay_days         |is_late|
+--------------------------------+--------------+----------+-----------------+------------+------------------+-------------------+-------+
|00010242fe8c5a6d1ba2dd792cb16214|RJ            |1         |72.19            |5.0         |7.614421296296296 |-8.01125           |0      |
|00048cc3ae777c65dbb7d2a0634bc1ea|MG            |1         |34.59            |4.0         |6.668067129629629 |-14.427372685185185|0      |
|0005a1a1728c9d785b8e2b08b904576c|SP            |1         |157.6            |1.0         |9.984004629629629 |0.7621643518518518 |1      |
|0008288aa423d2a3f00fcb17cd7d8719|SP            |2         |126.54           |5.0         |12.656261574074074|-7.419884259259259 |0      |
|000c3e6612759851cc3cbb4b83

## 6. Garanties Delta Lake : journal, Schema Enforcement

In [18]:
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé : écriture invalide refusée")
    print(str(e).splitlines()[0])
else:
    raise AssertionError("ATTENTION : l'écriture invalide a été acceptée")

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1795203756661,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"f477186b-bc61-4658-af02-82dba5e1eccd","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-63e0a09d-76dc-40e7-939b-62143e8dbd6c-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-3ed44a2c-dd3b-4686-8b12-ad4c0db0396c-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-c81846b3-9aab-4d8e-a066-5dc4ca793da0-c000.snappy.parquet","partition

## 7. MERGE INTO et Time Travel

Le MERGE de démonstration met à jour la même valeur sur 5 commandes existantes : il ne crée aucune fausse commande.

In [19]:
delta_table = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
updates = (spark.read.format("delta").load(SILVER_MAIN_PATH)
    .orderBy("order_id").limit(5))
(delta_table.alias("target")
    .merge(updates.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdate(set={"order_value": "source.order_value"})
    .execute())
print("MERGE INTO exécuté")

MERGE INTO exécuté


In [20]:
delta_table.history().select("version", "timestamp", "operation").show(truncate=False)
silver_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)
print("Version initiale :", silver_v0.count())
print("Version actuelle :", spark.read.format("delta").load(SILVER_MAIN_PATH).count())
print("J2 terminé. Vérifier les notes invalides avant de valider les KPI reviews.")

26/11/21 14:33:46 WARN MapPartitionsRDD: RDD 876 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting
                                                                                

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|3      |2026-11-21 14:33:34.233|MERGE    |
|2      |2026-11-21 14:32:46.808|WRITE    |
|1      |2026-11-20 19:44:31.805|MERGE    |
|0      |2026-11-20 19:42:39.162|WRITE    |
+-------+-----------------------+---------+



Version initiale : 99441


[Stage 517:===============================================>       (43 + 4) / 50]

Version actuelle : 99441
J2 terminé. Vérifier les notes invalides avant de valider les KPI reviews.
